# MPOPI + PPO trên mjlab (Google Colab, GPU)

Notebook này chạy nhánh `mpopi-ppo` của mjlab trên GPU của Colab:

1. Cài đặt môi trường (`uv`, PyTorch CUDA, MuJoCo Warp).
2. Chạy unit test của MPOPI.
3. Chạy thử nhanh (smoke test) trên GPU.
4. Benchmark A/B/C trên `Mjlab-Cartpole-Balance`, vẽ đồ thị và kiểm định thống kê.
5. (Tùy chọn) Train robot thật, ví dụ Unitree G1, với 3 chế độ: `ppo`, `naive_replay_ppo`, `mpopi_ppo`.
6. Xem TensorBoard và tải kết quả về.

**Trước khi chạy:** chọn *Runtime → Change runtime type → GPU*. T4 đủ cho Cartpole. Với G1 nên dùng L4 hoặc A100.

> **Lưu ý:** code MPOPI mới chỉ được kiểm thử trên CPU. Đây là lần đầu chạy trên GPU, nên hãy chạy mục 3 (smoke test) trước khi chạy các thí nghiệm dài.

## 0. Cấu hình

In [ ]:
#@title Cấu hình chung
#@markdown **Nguồn code:** `github` clone nhánh từ GitHub (phải push nhánh trước); `zip` upload file zip của repo.
SOURCE = "github"  #@param ["github", "zip"]
REPO_URL = "https://github.com/TamasTran/mjlab_MPOPI.git"  #@param {type:"string"}
BRANCH = "mpopi-ppo"  #@param {type:"string"}
#@markdown **Lưu kết quả lên Google Drive** (nên bật, vì Colab có thể ngắt kết nối và mất dữ liệu):
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

import os

REPO_DIR = "/content/mjlab_MPOPI"
if SAVE_TO_DRIVE:
  from google.colab import drive

  drive.mount("/content/drive")
  OUT_ROOT = "/content/drive/MyDrive/mpopi_results"
else:
  OUT_ROOT = "/content/mpopi_results"
os.makedirs(OUT_ROOT, exist_ok=True)
print("Kết quả sẽ lưu tại:", OUT_ROOT)

In [ ]:
#@title Hàm tiện ích: chạy lệnh và in log trực tiếp
import os
import subprocess
import sys


def run(cmd: str, cwd: str | None = None) -> None:
  """Run a shell command, stream its output, and raise if it fails."""
  print(f"$ {cmd}", flush=True)
  proc = subprocess.Popen(
    cmd,
    shell=True,
    cwd=cwd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ},
  )
  assert proc.stdout is not None
  for line in proc.stdout:
    print(line, end="", flush=True)
  if proc.wait() != 0:
    raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Lấy code

- `SOURCE = "github"`: clone nhánh `BRANCH`. Nhánh `mpopi-ppo` phải được push lên GitHub trước. Nếu repo private, dùng URL có token hoặc chọn cách zip.
- `SOURCE = "zip"`: upload file `mjlab_MPOPI_mpopi-ppo.zip` (tạo bằng `git archive`, xem README ở mục cuối).

In [ ]:
import os
import shutil
import zipfile

if os.path.exists(REPO_DIR):
  shutil.rmtree(REPO_DIR)

if SOURCE == "github":
  run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
else:
  from google.colab import files

  uploaded = files.upload()  # Chọn file mjlab_MPOPI_mpopi-ppo.zip
  name = next(iter(uploaded))
  with zipfile.ZipFile(name) as zf:
    zf.extractall("/content")
  assert os.path.isdir(REPO_DIR), f"Không thấy {REPO_DIR} sau khi giải nén"

assert os.path.isdir(f"{REPO_DIR}/src/mjlab/rl/mpopi"), "Code không có module MPOPI: sai nhánh?"
print("OK:", sorted(os.listdir(f"{REPO_DIR}/src/mjlab/rl/mpopi")))

## 3. Cài đặt môi trường

Cài `uv` rồi `uv sync` (PyTorch bản CUDA, MuJoCo Warp, RSL-RL 5.5.1). Lần đầu mất khoảng 5–10 phút.

Trên Colab dùng `uv sync` **không** kèm `--extra cpu` để có PyTorch bản CUDA.

In [ ]:
run("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
run("uv sync", cwd=REPO_DIR)
run(
  "uv run python -c \"import torch, warp, mujoco_warp, rsl_rl; "
  "print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), "
  "torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')\"",
  cwd=REPO_DIR,
)

## 4. Unit test của MPOPI

In [ ]:
run(
  "uv run pytest tests/test_mpopi_estimators.py tests/test_mpopi_replay.py "
  "tests/test_mpopi_algorithm.py -q -p no:warnings",
  cwd=REPO_DIR,
)

## 5. Smoke test trên GPU

Train Cartpole 5 vòng lặp ở chế độ `mpopi_ppo` bằng CLI `train` thật. Nếu bước này lỗi, đừng chạy các bước sau.

In [ ]:
run(
  "uv run train Mjlab-Cartpole-Balance --agent.logger tensorboard "
  "--env.scene.num-envs 512 --agent.max-iterations 5 "
  "--agent.algorithm.mpopi.mode mpopi_ppo --agent.algorithm.mpopi.replay-buffer-size 2 "
  f"--log-root {OUT_ROOT}/smoke",
  cwd=REPO_DIR,
)

## 6. Benchmark A/B/C trên Cartpole-Balance

Các nhánh so sánh:

| Nhánh | Ý nghĩa |
|---|---|
| `A_ppo` | PPO chuẩn |
| `A_ppo_bigmb` | PPO với số mẫu mỗi bước gradient bằng các nhánh replay (đối chứng cân bằng tính toán) |
| `B_naive_replay` | Replay dữ liệu cũ, coi như on-policy, không hiệu chỉnh |
| `C_mpopi` | Replay có hiệu chỉnh importance sampling (ρ̄ = 1) |
| `C_mpopi_noclip` | Như C nhưng không chặn trọng số |

Điểm số là reward trung bình mỗi bước của policy tất định trên env `play` riêng. Giá trị tối đa là 0,05 (1.0 sau chuẩn hóa).

Kết quả trên CPU (5 seed, xem `docs/mpopi_cartpole_results.md`) chưa đủ để kết luận. Trên GPU nên chạy **≥ 10 seed**. Nên chốt giả thuyết trước khi chạy.

In [ ]:
#@title Tham số benchmark
TASK = "Mjlab-Cartpole-Balance"  #@param {type:"string"}
NUM_ENVS = 256  #@param {type:"integer"}
SEEDS = 10  #@param {type:"integer"}
SEED_OFFSET = 300  #@param {type:"integer"}
ITERATIONS = 80  #@param {type:"integer"}
EVAL_EVERY = 5  #@param {type:"integer"}
ARMS = "A_ppo A_ppo_bigmb B_naive_replay C_mpopi C_mpopi_noclip"  #@param {type:"string"}
REPLAY_BUFFER_SIZE = 4  #@param {type:"integer"}
REPLAY_RATIO = 1.0  #@param {type:"number"}

BENCH_DIR = f"{OUT_ROOT}/bench_{TASK}_{NUM_ENVS}envs"

In [ ]:
run(
  "uv run python scripts/benchmarks/mpopi_benchmark.py "
  f"--task {TASK} --device cuda:0 --num-envs {NUM_ENVS} "
  f"--seeds {SEEDS} --seed-offset {SEED_OFFSET} --iterations {ITERATIONS} "
  f"--eval-every {EVAL_EVERY} --arms {ARMS} "
  f"--replay-buffer-size {REPLAY_BUFFER_SIZE} --replay-ratio {REPLAY_RATIO} "
  f"--out-dir {BENCH_DIR}",
  cwd=REPO_DIR,
)

### 6.1 Đồ thị và kiểm định theo cặp seed

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

MAX_REWARD = 0.05  # reward weight 1.0 x dt 0.05 cho Cartpole; đổi nếu dùng task khác.

df = pd.read_csv(f"{BENCH_DIR}/curves.csv")
ev = df[df["eval_return"].notna()].copy()
ev["score"] = ev["eval_return"] / MAX_REWARD

fig, ax = plt.subplots(figsize=(8, 4.5))
for arm, g in ev.groupby("arm"):
  stat = g.groupby("env_steps")["score"].agg(["mean", "std", "count"])
  ci = 1.96 * stat["std"] / stat["count"].pow(0.5)
  ax.plot(stat.index, stat["mean"], label=arm)
  ax.fill_between(stat.index, stat["mean"] - ci, stat["mean"] + ci, alpha=0.15)
ax.set_xlabel("Số bước môi trường")
ax.set_ylabel("Điểm chuẩn hóa (1.0 = tối đa)")
ax.set_title(f"{TASK}, {NUM_ENVS} envs, {SEEDS} seeds (mean ± 95% CI)")
ax.grid(alpha=0.3)
ax.legend()
fig.savefig(f"{BENCH_DIR}/curves.png", dpi=150, bbox_inches="tight")
plt.show()

summary = json.load(open(f"{BENCH_DIR}/summary.json"))
arms = [a for a in ARMS.split() if a in summary]
table = pd.DataFrame(
  {
    a: {
      "AUC": summary[a]["auc"]["mean"] / MAX_REWARD,
      "AUC ±95%": summary[a]["auc"]["ci95"] / MAX_REWARD,
      "final": summary[a]["final"]["mean"] / MAX_REWARD,
      "final ±95%": summary[a]["final"]["ci95"] / MAX_REWARD,
    }
    for a in arms
  }
).T
display(table.round(3))

# Kiểm định theo cặp: cùng seed = cùng khởi tạo mạng và môi trường.
rows = []
for a, b in [
  ("C_mpopi", "A_ppo"),
  ("C_mpopi", "A_ppo_bigmb"),
  ("C_mpopi", "B_naive_replay"),
  ("B_naive_replay", "A_ppo"),
  ("C_mpopi_noclip", "C_mpopi"),
]:
  if a not in summary or b not in summary:
    continue
  d = [
    (x - y) / MAX_REWARD
    for x, y in zip(summary[a]["per_seed"]["auc"], summary[b]["per_seed"]["auc"])
  ]
  p = stats.wilcoxon(d).pvalue if any(v != 0 for v in d) else math.nan
  rows.append(
    {
      "so sánh (AUC)": f"{a} − {b}",
      "chênh lệch TB": sum(d) / len(d),
      "số seed âm": f"{sum(v < 0 for v in d)}/{len(d)}",
      "Wilcoxon p": p,
    }
  )
display(pd.DataFrame(rows).round(4))

## 7. (Tùy chọn) Train robot với 3 chế độ

Chạy CLI `train` cho từng chế độ với cùng seed. Mặc định là `Mjlab-Velocity-Flat-Unitree-G1`.

- Mỗi lần train G1 có thể mất hàng giờ. Colab miễn phí có thể ngắt giữa chừng, nên bật `SAVE_TO_DRIVE` ở mục 0.
- Replay buffer không được lưu vào checkpoint: nếu resume, buffer bắt đầu lại từ rỗng.
- Log bằng TensorBoard để khỏi phải đăng nhập wandb.

In [ ]:
#@title Tham số train
ROBOT_TASK = "Mjlab-Velocity-Flat-Unitree-G1"  #@param {type:"string"}
ROBOT_NUM_ENVS = 2048  #@param {type:"integer"}
ROBOT_ITERATIONS = 500  #@param {type:"integer"}
ROBOT_SEED = 1  #@param {type:"integer"}
MODES = "ppo naive_replay_ppo mpopi_ppo"  #@param {type:"string"}

ROBOT_LOG_ROOT = f"{OUT_ROOT}/train_logs"

In [ ]:
for mode in MODES.split():
  run(
    f"uv run train {ROBOT_TASK} --agent.logger tensorboard "
    f"--env.scene.num-envs {ROBOT_NUM_ENVS} --agent.max-iterations {ROBOT_ITERATIONS} "
    f"--agent.seed {ROBOT_SEED} --agent.run-name {mode}_seed{ROBOT_SEED} "
    f"--agent.algorithm.mpopi.mode {mode} --log-root {ROBOT_LOG_ROOT}",
    cwd=REPO_DIR,
  )

## 8. TensorBoard

Các chỉ số cần xem:
- `Train/mean_reward`, `Train/mean_episode_length`.
- `Loss/kl`, `Loss/clip_fraction`: chỉ có ở các chế độ replay.
- `Loss/mpopi/ess`, `Loss/mpopi/weight_mean`, `Loss/mpopi/clipped_frac`, `Loss/mpopi/behavior_kl`, `Loss/mpopi/accepted`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUT_ROOT}

## 9. Tải kết quả về máy

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive("/content/mpopi_results", "zip", OUT_ROOT)
files.download(archive)

## Ghi chú

**Tạo file zip của nhánh** (trên máy local, trong thư mục repo), khi không muốn push lên GitHub:

```bash
git archive --format=zip --prefix=mjlab_MPOPI/ -o mjlab_MPOPI_mpopi-ppo.zip mpopi-ppo
```

**Ý nghĩa các chế độ `--agent.algorithm.mpopi.mode`:**
- `ppo`: PPO chuẩn của RSL-RL, không thay đổi gì.
- `naive_replay_ppo`: thêm dữ liệu cũ vào batch PPO như thể on-policy.
- `mpopi_ppo`: thêm dữ liệu cũ, có hiệu chỉnh importance sampling (trọng số `clip(π_old/μ)`, V-trace).